## Analyse identified with scPRINT regulatory gene networks in Fetal Stem Cells

* **Author**: Anna Maguza 
* **Location**: CellZome, GSK company
* **Creation date**: 04.09.2025
* **Last modified date**: 08.09.2025

In this notebook, we perform an extensive analysis of gene networks generated by scPRINT in our previous notebook for fetal stem cells. 

We can look at many patterns from the gene networks such as hub genes, gene modules, and gene-gene interactions. Some of them are related to highly variable and differentially expressed genes while others can only be inferred by using the gene networks.

Table of content:

1. Hub genes
2. Network communities
3. Heatmaps visualizing network communities
4. Gene networks
5. Gene set enrichment analysis
6. Comparing with add-on programs from NichCompass analysis

In [1]:
from bengrn import BenGRN
import scanpy as sc
from bengrn.base import train_classifier

from anndata.utils import make_index_unique
from grnndata import utils as grnutils
from grnndata import read_h5ad

from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

from scdataloader import utils as data_utils
import numpy as np

from pyvis import network as pnx
import networkx as nx
import scipy.sparse
import pandas as pd
import gseapy as gp
from gseapy import dotplot

import os
import ssl
import urllib3

%load_ext autoreload
%autoreload 2 

/Users/am336941/uv_envs/scprint_env230/scprint_env230/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


→ connected lamindb: anonymous/test


In [2]:
adata = sc.read_h5ad("gut_data/gut_hs_full_annotated_AM_06032025_140458_raw.h5ad")
adata

AnnData object with n_obs × n_vars = 365542 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_pro

In [3]:
fetal_mask = ["second trimester", "first trimester"]
adata = adata[adata.obs["age_group"].isin(fetal_mask)]

In [4]:
adata.obs["age_group"].value_counts()

age_group
second trimester    169380
first trimester     148150
Name: count, dtype: int64

In [5]:
adata.obs

,cell_index,Source Name,ENA_SAMPLE,BioSD_SAMPLE,organism,disease,organism_part,cell_type,growth_condition,developmental_stage,Material Type,Protocol REF,sample_id,LIBRARY_LAYOUT,cdna_read_size,cell_barcode_size,end_bias,library_construction,sample_barcode_size,umi_barcode_offset,umi_barcode_size,Performer,Assay Name,ENA_EXPERIMENT,ENA_RUN,time,time_unit,n_genes,doublet_scores,predicted_doublets,n_counts,log1p_n_counts,log1p_n_genes,percent_mito,n_counts_mito,percent_ribo,n_counts_ribo,percent_hb,n_counts_hb,percent_top50,cell_passed_qc,qc_cluster,cluster_passed_qc,consensus_fraction,consensus_passed_qc,total_counts,n_genes_by_counts,percent_chrY,XIST-counts,XIST-percentage,sex,S_score,G2M_score,Cell_cycle_phase,Study_name,ArrayExpress_ID,library_preparation_protocol,full_age,age_group,immunophenotype,metadata_cluster,barcode,category,Integrated_05,age,gestational_age,donor_id,passage,batch,sampling_site,celltype,library_construnction_and_layout,cell_states,cellstates_scANVI,confidence_score,gut_region,cell_state
cell_index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
AAACGGGTCGAATCCA-1-4918STDY7421299_ERR4010667,AAACGGGTCGAATCCA-1-4918STDY7421299_ERR4010667,BRC2043_CO_pos_cells,ERS4414933,SAMEA6655464,Homo sapiens,normal,colon,intestinal epithelial cell,primary tissue,10th week post-fertilization human stage,cell,P-MTAB-95162,BRC2043_CO_pos_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,BRC2043_CO_pos_cells,ERX4012067,ERR4010667,N/A,week,2373,0.186747,False,9640,9.173780,7.772332,1.649378,159.0,35.041494,3378.0,0.0,0.0,36.960581,True,8,True,0.6,True,9640,2373,0.103734,0,0.000000,male,-0.014920,-0.278880,G1,Elementaite_2021,E-MTAB-8901,10xV2 3 prime tag,12.2 week,second trimester,EPCAM positive,3,AAACGGGTCGAATCCA,Epithelial,Colonocyte,,12.2,BRC2043,N/A,4918STDY7421299,unknown,Epithelial,10xV2_PAIRED,Stem cells,Stem cells,0.318226,large intestine,NaN
AAAGCAACAAACCTAC-1-4918STDY7421299_ERR4010667,AAAGCAACAAACCTAC-1-4918STDY7421299_ERR4010667,BRC2043_CO_pos_cells,ERS4414933,SAMEA6655464,Homo sapiens,normal,colon,intestinal epithelial cell,primary tissue,10th week post-fertilization human stage,cell,P-MTAB-95162,BRC2043_CO_pos_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,BRC2043_CO_pos_cells,ERX4012067,ERR4010667,N/A,week,5863,0.163934,False,61106,11.020382,8.676587,4.670572,2854.0,35.659346,21790.0,0.0,0.0,39.591857,True,8,True,1.0,True,61106,5863,0.132557,0,0.000000,male,-0.254676,-1.023004,G1,Elementaite_2021,E-MTAB-8901,10xV2 3 prime tag,12.2 week,second trimester,EPCAM positive,3,AAAGCAACAAACCTAC,Epithelial,Colonocyte,,12.2,BRC2043,N/A,4918STDY7421299,unknown,Epithelial,10xV2_PAIRED,Colonocyte,Colonocyte,0.986588,large intestine,NaN
AAATGCCAGGGATCTG-1-4918STDY7421299_ERR4010667,AAATGCCAGGGATCTG-1-4918STDY7421299_ERR4010667,BRC2043_CO_pos_cells,ERS4414933,SAMEA6655464,Homo sapiens,normal,colon,intestinal epithelial cell,primary tissue,10th week post-fertilization human stage,cell,P-MTAB-95162,BRC2043_CO_pos_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,BRC2043_CO_pos_cells,ERX4012067,ERR4010667,N/A,week,454,0.250000,False,965,6.873164,6.120297,8.082902,78.0,41.347150,399.0,0.0,0.0,47.150259,True,15,True,0.8,True,965,454,0.000000,0,0.000000,male,-0.174000,0.495131,G2M,Elementaite_2021,E-MTAB-8901,10xV2 3 prime tag,12.2 week,second trimester,EPCAM positive,3,AAATGCCAGGGATCTG,Unknown,Unknown,,12.2,BRC2043,N/A,4918STDY7421299,unknown,Epithelial,10xV2_PAIRED,Enterocyte,Enterocyte,0.694930,large intestine,NaN
AACCGCGTCATTTGGG-1-4918STDY7421299_ERR4010667,AACCGCGTCATTTGGG-1-4918STDY7421299_ERR4010667,BRC2043_CO_pos_cells,ERS4414933,SAMEA6655464,Homo sapiens,normal,colon,intestinal epithelial cell,primary tissue,10th week post-fertilization human stage,cell,P-MTAB-95162,BRC2043_CO_pos_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,BRC2043_CO_pos_cells,ERX4012067,ERR4010667,N/A,week,2538,0.186747,False,10879

## Processing GRN Object for Analysis

Let's process the grn_full object similar to the cancer tutorial to enable detailed network analysis.

In [ ]:
# load the generated gene networks from the previous notebook
grn_full = read_h5ad("/1_scPRINT/data/grn_sc_fetal.h5ad")

In [7]:
# Remove duplicates and process gene symbols
grn_full.var.symbol = make_index_unique(grn_full.var.symbol.astype(str))

# Convert gene ids to symbols for better readability
grn_full.var['ensembl_id'] = grn_full.var.index
grn_full.var.index = grn_full.var.symbol

print(f"GRN shape: {grn_full.shape}")
print(f"Number of genes: {grn_full.n_vars}")
print(f"Number of cells/observations: {grn_full.n_obs}")

GRN shape: (7305, 3000)
Number of genes: 3000
Number of cells/observations: 7305


In [8]:
grn_full.obs

,cell_index,Source Name,ENA_SAMPLE,BioSD_SAMPLE,organism,disease,organism_part,cell_type,growth_condition,developmental_stage,Material Type,Protocol REF,sample_id,LIBRARY_LAYOUT,cdna_read_size,cell_barcode_size,end_bias,library_construction,sample_barcode_size,umi_barcode_offset,umi_barcode_size,Performer,Assay Name,ENA_EXPERIMENT,ENA_RUN,time,time_unit,n_genes,doublet_scores,predicted_doublets,n_counts,log1p_n_counts,log1p_n_genes,percent_mito,n_counts_mito,percent_ribo,n_counts_ribo,percent_hb,n_counts_hb,percent_top50,cell_passed_qc,qc_cluster,cluster_passed_qc,consensus_fraction,consensus_passed_qc,total_counts,n_genes_by_counts,percent_chrY,XIST-counts,XIST-percentage,sex,S_score,G2M_score,Cell_cycle_phase,Study_name,ArrayExpress_ID,library_preparation_protocol,full_age,age_group,immunophenotype,metadata_cluster,barcode,category,Integrated_05,age,gestational_age,donor_id,passage,batch,sampling_site,celltype,library_construnction_and_layout,cell_states,cellstates_scANVI,confidence_score,gut_region,cell_state,organism_ontology_term_id,cell_type_ontology_term_id,development_stage_ontology_term_id,tissue_ontology_term_id,sex_ontology_term_id,assay_ontology_term_id,self_reported_ethnicity_ontology_term_id,disease_ontology_term_id,cell_culture,nnz,log1p_n_genes_by_counts,log1p_total_counts,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb,outlier,mt_outlier,batches
7c8ca1f2-2d4c-4381-8af0-3e1449e60412,GGTGAAGGTTATCACG-1-Human_colon_16S7985395_ERR4...,Human_colon_16S7985395,ERS5054879,SAMEA7296313,Homo sapiens,normal,large intestine,unsorted,primary tissue,embryo,organism part,P-MTAB-100897,Human_colon_16S7985395,PAIRED,98,16,5 prime tag,10xV2,8,16,10,"Rasa Elmentaite, Kylie James",Human_colon_16S7985395,ERX4509938,ERR4574281,N/A,week,8239,0.191837,False,553602,13.224203,9.016877,5.784661,32024.0,49.054917,271569.0,0.010838,60.0,48.016264,True,42,True,1.0,True,553241.0,8182,0.000000,28,0.005058,female,1.109082,1.069114,S,Elementaite_2021,E-MTAB-9536,10xV2 5 prime tag,12 week,first trimester,unsorted,2,GGTGAAGGTTATCACG,Unknown,Unknown,12,N/A - not fetal,F67,N/A,Human_colon_16S7985395,unknown,Epithelial,10xV2_PAIRED,Stem cells,Stem cells,0.676587,large intestine,NaN,NCBITaxon:9606,CL:1001566,HsapDv:0000045,UBERON:0000059,PATO:0000383,EFO:0009900,unknown,PATO:0000461,False,8239,9.009814,13.223551,26.204674,32024.0,10.374272,5.788436,271564.0,12.511957,49.086022,60.0,4.110874,0.010845,True,False,"EFO:0009900,unknown,PATO:0000383,F67"
d5150c76-e56e-41f0-ad7d-34876999ccb4,TTGCCGTTCCTAGGGC-1-Human_colon_16S7985391_ERR4...,Human_colon_16S7985391,ERS5054890,SAMEA7296324,Homo sapiens,normal,ileum,unsorted,primary tissue,embryo,organism part,P-MTAB-100897,Human_colon_16S7985391,PAIRED,98,16,5 prime tag,10xV2,8,16,10,"Rasa Elmentaite, Kylie James",Human_colon_16S7985391,ERX4509949,ERR4574292,N/A,week,8922,0.143876,False,431037,12.973952,9.096387,7.423725,31999.0,44.047727,189862.0,0.021112,91.0,42.335345,True,42,True,1.0,True,430606.0,8858,0.106255,0,0.000000,male,1.749627,1.165088,S,Elementaite_2021,E-MTAB-9536,10xV2 5 prime tag,15 week,second trimester,unsorted,2,TTGCCGTTCCTAGGGC,Unknown,Unknown,15,N/A - not fetal,F66,N/A,Human_colon_16S7985391,proximal,Epithelial,10xV2_PAIRED,Stem cells,Stem cells,0.770936,small intestine,NaN,NCBITaxon:9606,CL:1001566,HsapDv:0000046,UBERON:0002108,PATO:0000384,EFO:0009900,unknown,PATO:0000461,False,8922,9.089189,12.972951,22.823184,31999.0,10.373491,7.431155,189860.0,12.154048,44.091350,91.0,4.521789,0.021133,True,False,"EFO:0009900,unknown,PATO:0000384,F66"
a9a0227f-915f-4834-b2cd-1eece0e26188,TTGTAGGGTGTTTGTG-1-Human_colon_16S7985390_ERR4...,Human_colon_16S7985390,ERS5054885,SAMEA7296319,Homo sapiens,normal,ileum,unsorted,primary tissue,embryo,organism part,P-MTAB-100897,Human_colon_16S7985390,PAIRED,98,16,5 prime tag,10xV2,8,16,10,"Rasa Elmentaite, Kylie James",Hum

In [9]:
network_matrix = grn_full.grn

In [10]:
network_matrix

symbol,TSPAN6,DPM1,FUCA2,CFTR,LAP3,ICA1,DBNDD1,SLC7A2,ARF5,SARM1,POLDIP2,AK2,FKBP4,KDM1A,NDUFAB1,HCCS,SLC25A5,POLR2J,RPAP3,ACSM3,SPPL2B,PDK2,GDE1,REX1BD,SPATA20,BAIAP2L1,USH1C,GGCT,ELAC2,ARSD,PNPLA4,SLC13A2,TRAPPC6A,RPUSD1,TSR3,SYPL1,CYB561,PLEKHG6,SS18L2,MPND,MED24,RPS20,CSDE1,VTA1,UQCRC1,CD9,FUZ,LYPLA2,MKS1,UTP18,PTBP1,SLC25A39,NUB1,HEBP1,UFL1,CAPN1,MDH1,SLC30A9,MTMR11,CCDC88C,DPEP1,BID,CHDH,SLC38A5,RALBP1,TTC27,PLEKHB1,SERPINB1,CPS1,ERP44,DERA,STRAP,NR1H3,TOMM34,SEC63,MIPEP,TFB1M,HMGB3,SARS1,EIPR1,ALG1,FUT8,MAP2K3,TMSB10,SH3YL1,FAM136A,RFC1,CUL3,OTC,RPL26L1,NSUN2,BOD1L1,MAT2B,CDH1,MTREX,C6,PNKP,PHF23,PSMA4,LSG1,...,CCDC85C,HMGN1,LCMT1,RNPS1,DNAJC19,TSN,MT-ND4L,DYNLT2B,QTRT1,VDAC1,NDUFS3,RPS29,NUDT19,AP1G2,CPNE1,LYRM4,ALG3,SMIM30,FIS1,STARD10,IFRD2,CSKMT,PHB2,MYL5,TSTD1,ZNF579,PPP2R2A,CCNL2,UBA52,EXOSC6,HNRNPA1L3,NDUFAF8,NOL7,TMEM185B,HSBP1L1,MT-ATP8,FAM174C,OST4,RPL41,ZNF688,ACBD6,HSBP1,TMA7,MCTS1,PHGR1,TMEM238,RPS28,FAM133B,HGH1,CDKN2AIPNL,TMEM250,NME1,ADSL,PPIL3,RDH14,RPL36A,ATP5MF,ATP5PO,PISD,EIF6,MRPL20,DDOST,H2AJ,ADH1C,ABHD14A,SMIM20,C1orf210,EIF5AL1,ATXN7L3B,PRKDC,EEF1G,BRK1,HMBS,POLG2,GATC,CNPY2,SMIM6,RBM15B,BOP1,SSBP1,SRSF8,RBM8A,TIMM23,RPL17,MRPS21,SMIM22,NCBP2AS2,CD24,DCP1A,PPP4R3B,MCCC2,UHRF1,DACH1,GSTT1,nan,SCO2,nan-1,PRRC2B,HOMEZ,SOD2
symbol,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
TSPAN6,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.019609,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.041953,0.000000,0.000870,0.00000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.031259,0.000000,0.000000
DPM1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.044727,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000344,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.044395,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002025,0.001034,0.000000,0.000000,0.000000,0.031274,0.000000,0.000000
FUCA2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.049958,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,

## Hub Genes Analysis

We'll analyze hub genes using degree centrality (number of connections) and eigenvector centrality.

In [11]:
# Hub genes based on edge centrality (number of connections / strength of connections)
print("Top 20 hub genes by connection strength:")
top_hubs = grn_full.grn.sum(0).sort_values(ascending=False).head(20)
print(top_hubs)

Top 20 hub genes by connection strength:
symbol
RPL26L1     133.039322
EIF5AL1     119.376427
PRRC2B       90.598045
MRPL36       47.252304
TAMM41       37.780815
POLR1B       30.571976
SENP6        30.004679
ALG6         29.909336
DACH1        25.744516
MRPS11       25.636396
TSPAN8       23.574320
EIF1AY       17.895508
ZNF148       17.836872
BPTF         17.007496
POP7         15.735017
KRT18        15.183529
HID1         13.920466
CRACR2B      13.654858
TMEM126A     13.169231
HPCAL1       12.779634
dtype: float32


In [12]:
# Compute eigenvector centrality by creating a sparse network with top 20 neighbors per gene
TOP = 20

print("Computing centrality measures...")
grnutils.get_centrality(grn_full, TOP, top_k_to_disp=0)

print("Top 20 genes by eigenvector centrality:")
top_centrality = grn_full.var.centrality.sort_values(ascending=False).head(20)
print(top_centrality)

Computing centrality measures...
Top central genes: []
Top 20 genes by eigenvector centrality:
symbol
RPL26L1     0.269750
EIF5AL1     0.268328
TAMM41      0.266374
ALG6        0.264062
MRPL36      0.256255
SENP6       0.241618
MRPS11      0.239160
BPTF        0.227616
POLR1B      0.226165
TSPAN8      0.219002
PRRC2B      0.212380
POP7        0.207199
CRACR2B     0.206828
ZNF148      0.191724
DACH1       0.189337
TMEM126A    0.172570
HID1        0.169814
KRT18       0.165953
THOC2       0.135917
GUF1        0.132812
Name: centrality, dtype: float64


## Network Communities Detection

Using Leiden algorithm to find communities in the network, focusing on communities between 20-200 genes.

In [13]:
grn_full = grnutils.compute_cluster(grn_full, resolution=1.5, use="leiden", n_iterations=10, max_comm_size=200)

In [14]:
grn_full.var['cluster_1.5'].value_counts()

cluster_1.5
0     200
6     200
11    200
9     200
8     200
7     200
10    200
5     200
4     200
3     200
2     200
1     200
12    165
13    152
14    146
15     76
16     18
17     10
18      5
19      4
21      3
20      3
22      1
23      1
38      1
37      1
36      1
35      1
34      1
33      1
32      1
31      1
30      1
29      1
28      1
27      1
26      1
25      1
24      1
39      1
Name: count, dtype: int64

## Create a hierarchically-clustered heatmap

In [15]:
# delete RPL26L1 
#network_matrix = network_matrix[network_matrix.index != 'PRRC2B']
#grn_full.var['cluster_1.5'] = grn_full.var['cluster_1.5'][grn_full.var['cluster_1.5'] != 'PRRC2B']

In [15]:
# Set high DPI for better quality
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [16]:
# Get unique communities and filter out small ones
unique_communities = grn_full.var['cluster_1.5'].unique()
print(f"All communities found: {sorted(unique_communities)}")

All communities found: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '5', '6', '7', '8', '9']


In [17]:
# Filter communities with at least 100 genes
valid_communities = []
community_gene_counts = {}

for community in unique_communities:
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    community_gene_counts[community] = len(community_genes)
    if len(community_genes) >= 100:
        valid_communities.append(community)

In [18]:
# Select top N genes from each valid community
selected_genes = []
community_labels = []
top_n = 10 
for community in sorted(valid_communities):
    # Get genes in this community
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    
    # Calculate gene strengths (total interaction strength) for genes in this community
    gene_strengths = []
    for gene in community_genes:
        if gene in network_matrix.index:
            # Calculate total absolute interaction strength for this gene
            gene_total_strength = network_matrix.loc[gene, :].abs().sum()
            gene_strengths.append((gene, gene_total_strength))
        else:
            gene_strengths.append((gene, 0))  # Missing genes get 0 strength
    
    # Sort by strength (descending) and select top N
    gene_strengths.sort(key=lambda x: x[1], reverse=True)
    selected_community_genes = [gene for gene, strength in gene_strengths[:top_n]]
    
    selected_genes.extend(selected_community_genes)
    community_labels.extend([community] * len(selected_community_genes))

In [19]:
# Check which genes exist in network_matrix
missing_genes = []
valid_genes = []

for gene in selected_genes:
    if gene in network_matrix.index and gene in network_matrix.columns:
        valid_genes.append(gene)
    else:
        missing_genes.append(gene)

print(f"Valid genes in network_matrix: {len(valid_genes)}")
print(f"Missing genes: {len(missing_genes)}")

Valid genes in network_matrix: 150
Missing genes: 0


In [20]:
# Filter network matrix to selected genes
filtered_matrix = network_matrix.loc[valid_genes, valid_genes]

# Update community labels for valid genes only
valid_community_labels = []
for gene in valid_genes:
    original_index = selected_genes.index(gene)
    valid_community_labels.append(community_labels[original_index])

print(f"\nFinal matrix shape: {filtered_matrix.shape}")


Final matrix shape: (150, 150)


In [21]:
# Calculate community-level interaction matrices for clustering
community_interaction_matrix = pd.DataFrame(index=sorted(valid_communities), 
                                           columns=sorted(valid_communities))

for comm1 in sorted(valid_communities):
    for comm2 in sorted(valid_communities):
        # Get genes from each community
        genes_comm1 = [gene for i, gene in enumerate(valid_genes) 
                      if valid_community_labels[i] == comm1]
        genes_comm2 = [gene for i, gene in enumerate(valid_genes) 
                      if valid_community_labels[i] == comm2]
        
        # Calculate mean interaction between communities
        if genes_comm1 and genes_comm2:
            interaction_submatrix = filtered_matrix.loc[genes_comm1, genes_comm2]
            mean_interaction = interaction_submatrix.values.mean()
            community_interaction_matrix.loc[comm1, comm2] = mean_interaction
        else:
            community_interaction_matrix.loc[comm1, comm2] = 0

# Convert to numeric
community_interaction_matrix = community_interaction_matrix.astype(float)

In [22]:
community_linkage = linkage(pdist(community_interaction_matrix, metric='euclidean'), method='ward')

In [23]:
# Get community ordering from clustering
community_dendro = dendrogram(community_linkage, no_plot=True)
community_order = [sorted(valid_communities)[i] for i in community_dendro['leaves']]

print(f"Community clustering order: {community_order}")

Community clustering order: ['13', '4', '7', '10', '3', '1', '2', '12', '11', '9', '0', '6', '5', '14', '8']


In [24]:
# Reorder genes based on community clustering
reordered_genes = []
reordered_community_labels = []

for community in community_order:
    community_genes = [gene for i, gene in enumerate(valid_genes) 
                      if valid_community_labels[i] == community]
    reordered_genes.extend(community_genes)
    reordered_community_labels.extend([community] * len(community_genes))

In [25]:
# Reorder matrix
gene_to_index = {gene: i for i, gene in enumerate(valid_genes)}
new_indices = [gene_to_index[gene] for gene in reordered_genes]
reordered_matrix = filtered_matrix.iloc[new_indices, new_indices]

In [26]:
# Community colors
community_colors = plt.cm.tab20(np.linspace(0, 1, len(valid_communities)))
community_color_map = {comm: color for comm, color in zip(sorted(valid_communities), community_colors)}

In [ ]:
# Create figure with gridspec for better control
fig = plt.figure(figsize=(24, 20))
gs = GridSpec(4, 4, figure=fig, height_ratios=[1.2, 0.05, 3, 0.3], width_ratios=[0.3, 0.05, 3, 0.8])

# Top dendrogram - showing community clustering
ax_dendro_top = fig.add_subplot(gs[0, 2])
dendro_top = dendrogram(community_linkage, ax=ax_dendro_top, orientation='top', 
                       labels=[f'Comm {c}' for c in sorted(valid_communities)],
                       color_threshold=0, leaf_font_size=10)
ax_dendro_top.set_title('Community Similarity Dendrogram', fontsize=12, fontweight='bold', pad=10)
ax_dendro_top.set_xticks([])
ax_dendro_top.tick_params(axis='y', labelsize=8)

# Left dendrogram - same community clustering  
ax_dendro_left = fig.add_subplot(gs[2, 0])
dendro_left = dendrogram(community_linkage, ax=ax_dendro_left, orientation='left',
                        labels=[f'Comm {c}' for c in sorted(valid_communities)],
                        color_threshold=0, leaf_font_size=10)
ax_dendro_left.set_ylabel('Community Similarity Dendrogram', fontsize=12, fontweight='bold')
ax_dendro_left.set_yticks([])
ax_dendro_left.tick_params(axis='x', labelsize=8)

# Main heatmap
ax_heatmap = fig.add_subplot(gs[2, 2])
vmax = max(abs(reordered_matrix.min().min()), abs(reordered_matrix.max().max()))
vmin = -vmax

im = ax_heatmap.imshow(reordered_matrix, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)

# Set gene labels on axes with smaller font
ax_heatmap.set_xticks(range(len(reordered_genes)))
ax_heatmap.set_yticks(range(len(reordered_genes)))
ax_heatmap.set_xticklabels(reordered_genes, rotation=90, ha='center', fontsize=3)
ax_heatmap.set_yticklabels(reordered_genes, fontsize=3)

# Remove tick marks but keep labels
ax_heatmap.tick_params(axis='both', which='both', length=0, width=0, pad=1)

# Community color bar on the left
ax_community = fig.add_subplot(gs[2, 1])
community_colors_reordered = [community_color_map[label] for label in reordered_community_labels]
community_array = np.array(community_colors_reordered).reshape(-1, 1)
ax_community.imshow(community_array, aspect='auto', cmap=ListedColormap(community_colors_reordered))
ax_community.set_xticks([])
ax_community.set_yticks([])
ax_community.set_ylabel('Communities', fontsize=12, rotation=90, fontweight='bold')

# Community color bar on the top
ax_community_top = fig.add_subplot(gs[1, 2])
community_array_top = np.array(community_colors_reordered).reshape(1, -1)
ax_community_top.imshow(community_array_top, aspect='auto', cmap=ListedColormap(community_colors_reordered))
ax_community_top.set_xticks([])
ax_community_top.set_yticks([])
ax_community_top.set_xlabel('Communities', fontsize=12, fontweight='bold')

# Add community boundaries
current_pos = 0
for community in community_order:
    community_size = reordered_community_labels.count(community)
    if community_size > 0:
        # Add thin white lines to separate communities
        if current_pos > 0:
            ax_heatmap.axhline(y=current_pos-0.5, color='white', linewidth=1)
            ax_heatmap.axvline(x=current_pos-0.5, color='white', linewidth=1)
        current_pos += community_size

# Colorbar
gs = GridSpec(4, 4, figure=fig, height_ratios=[1.2, 0.05, 3, 0.3], width_ratios=[0.3, 0.05, 3, 0.1])
ax_cbar = fig.add_subplot(gs[2, 3])
cbar = plt.colorbar(im, cax=ax_cbar)
cbar.set_label('Gene Interaction Strength', fontsize=12, fontweight='bold')
cbar.ax.tick_params(labelsize=10)

# Add legend
legend_patches = []
for community in community_order:  # Use clustered order
    color = community_color_map[community]
    count = reordered_community_labels.count(community)
    legend_patches.append(mpatches.Patch(color=color, label=f'Community {community} ({count} genes)'))

ax_legend = fig.add_subplot(gs[3, :])
ax_legend.legend(handles=legend_patches, loc='center', ncol=min(5, len(valid_communities)), fontsize=11)
ax_legend.axis('off')

plt.savefig("/1_scPRINT/fetalSC_figures/community_heatmaps/community_clustering_results.png", bbox_inches='tight', dpi=300)

## Create a hierarchically-clustered heatmap for each community

In [ ]:
# Set high DPI for better quality
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# Get unique communities
unique_communities = grn_full.var['cluster_1.5'].unique()
print(f"All communities found: {sorted(unique_communities)}")

# Filter communities with at least 40 genes
valid_communities_for_heatmaps = []
community_gene_counts = {}

for community in unique_communities:
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    community_gene_counts[community] = len(community_genes)
    if len(community_genes) >= 40:
        valid_communities_for_heatmaps.append(community)

# Create directory for saving plots if it doesn't exist
output_dir = "community_heatmaps"
os.makedirs(output_dir, exist_ok=True)

# Create individual heatmaps for each valid community
for community in sorted(valid_communities_for_heatmaps):
    
    # Get all genes in this community
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    
    # Filter to genes that exist in network_matrix
    valid_community_genes = [gene for gene in community_genes 
                            if gene in network_matrix.index and gene in network_matrix.columns]
    
    if len(valid_community_genes) < 5:
        print(f"  SKIPPED: Too few valid genes ({len(valid_community_genes)})")
        continue
    
    # Extract submatrix for this community
    community_matrix = network_matrix.loc[valid_community_genes, valid_community_genes]
    
    # Calculate matrix statistics
    non_zero_interactions = (community_matrix != 0).sum().sum()
    total_interactions = community_matrix.size
    matrix_density = non_zero_interactions / total_interactions
    
    # Determine figure size based on number of genes
    base_size = max(8, min(20, len(valid_community_genes) * 0.3))
    figsize = (base_size, base_size)
    
    # Create clustermap with dendrogram
    try:
        # Calculate gene label font size based on number of genes
        if len(valid_community_genes) <= 50:
            label_fontsize = 6
        elif len(valid_community_genes) <= 100:
            label_fontsize = 4
        else:
            label_fontsize = 3
        
        # Calculate clustering once and apply to both axes
        linkage_matrix = linkage(pdist(community_matrix, metric='euclidean'), method='ward')
        
        # Create clustermap with same clustering for rows and columns
        g = sns.clustermap(community_matrix,
                          row_linkage=linkage_matrix,
                          col_linkage=linkage_matrix,  # Use same linkage for both axes
                          figsize=figsize,
                          cmap='RdBu_r',
                          center=0,
                          square=True,
                          linewidths=0.01,
                          cbar_kws={"shrink": 0.8, "label": "Interaction Strength"},
                          xticklabels=True,
                          yticklabels=True)
        
        # Get the clustered gene order
        row_order = g.dendrogram_row.reordered_ind
        clustered_genes = [valid_community_genes[i] for i in row_order]
        
        # Ensure both axes have the same gene labels in the same order
        g.ax_heatmap.set_xticklabels(clustered_genes, 
                                    rotation=90, ha='center', fontsize=label_fontsize)
        g.ax_heatmap.set_yticklabels(clustered_genes, 
                                    rotation=0, fontsize=label_fontsize)
        
        # Remove tick marks but keep labels
        g.ax_heatmap.tick_params(axis='both', which='both', length=0, width=0, pad=1)
        
        # Add title
        g.fig.suptitle(f'Community {community} - Internal Gene Interactions\n'
                      f'{len(valid_community_genes)} genes, {matrix_density:.1%} density\n'
                      f'Range: [{community_matrix.min().min():.3f}, {community_matrix.max().max():.3f}]',
                      fontsize=14, fontweight='bold', y=0.98)
        
        # Save the plot
        filename = f"community{community}.png"
        filepath = os.path.join(output_dir, filename)
        g.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  ✓ Saved: {filepath}")
        
        # Close the figure to free memory
        plt.close(g.fig)

    except Exception as e:
        print(f"  ✗ Error creating heatmap for community {community}: {str(e)}")
        continue

## Gene networks

In [30]:
grn_full.var['cluster_1.5']

symbol
TSPAN6     3
DPM1       4
FUCA2      1
CFTR       5
LAP3       7
          ..
SCO2      13
nan-1      4
PRRC2B     4
HOMEZ      4
SOD2      13
Name: cluster_1.5, Length: 3000, dtype: category
Categories (40, object): ['0', '1', '2', '3', ..., '36', '37', '38', '39']

In [31]:
community_counts = grn_full.var['cluster_1.5'].value_counts()

In [ ]:
# delete communities with less than 40 genes
min_genes = 40
valid_communities = community_counts[community_counts >= min_genes].index

for community in valid_communities:
    selected_community = community
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == selected_community].index.tolist()
    
    # Create a subgraph for this specific community
    G = grn_full.plot_subgraph(community_genes, only=100, interactive=False)
    
    plt.figure(figsize=(12, 10))

    # Use a layout algorithm for better visualization
    pos = nx.spring_layout(G, k=1, iterations=50)

    # Draw the network
    nx.draw(G, pos,
            with_labels=True,
            node_color='lightblue',
            node_size=500,
            font_size=8,
            font_weight='bold',
            arrows=True,
            edge_color='gray',
            alpha=0.7)

    plt.title(f'Gene Regulatory Network - Community {selected_community}',
              fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save as PNG
    plt.savefig(f"/1_scPRINT/grn_community_{selected_community}.png",
                dpi=300, bbox_inches='tight')
    plt.close()

## Gene set enrichment analysis

In [33]:
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Set environment variables to disable SSL verification
os.environ['PYTHONHTTPSVERIFY'] = '0'
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['SSL_VERIFY'] = 'false'

# Configure SSL context
ssl._create_default_https_context = ssl._create_unverified_context

# Configure requests to not verify SSL
import requests.packages.urllib3.util.ssl_
requests.packages.urllib3.util.ssl_.DEFAULT_CIPHERS = 'ALL'

# Monkey patch requests to disable SSL verification
original_request = requests.Session.request
def patched_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return original_request(self, method, url, **kwargs)
requests.Session.request = patched_request


In [ ]:
# delete communities with less than 40 genes
min_genes = 40
valid_communities = community_counts[community_counts >= min_genes].index

for community in valid_communities:
    selected_community = community
    grn_full.get(grn_full.var[grn_full.var['cluster_1.5']== selected_community].index.tolist()).grn.sum(0).sort_values(ascending=False)
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == selected_community].index.tolist()
    
    try:
        # Use community-specific genes for enrichment analysis
        enr = gp.enrichr(gene_list=community_genes,  # Changed from list(G.nodes) to community_genes
                        gene_sets=['KEGG_2021_Human', 'MSigDB_Hallmark_2020', 'Reactome_2022', 
                        'Tabula_Sapiens', 'WikiPathway_2023_Human', 'TF_Perturbations_Followed_by_Expression', 
                        'Reactome', 'PPI_Hub_Proteins', 'OMIM_Disease', 'GO_Molecular_Function_2023'],
                        organism='Human',
                        background=grn_full.var.symbol.tolist())

        enr.res2d.Term = enr.res2d.Term.str.split("(").str[0].str[:50]
        ax = dotplot(enr.res2d.replace([np.inf, -np.inf], 0).dropna(),
                    column="Adjusted P-value",
                    title=f'Fetal Stem Cells Community {selected_community}',
                    cmap=plt.cm.viridis,
                    size=1,
                    top_term=15,
                    figsize=(7,7), 
                    cutoff=0.25)
        
        # Reduce title and axis font sizes
        plt.title(f'Fetal Stem Cells Community {selected_community}', fontsize=10)
        plt.xticks(fontsize=7)
        plt.yticks(fontsize=7)
        
        plt.tight_layout()
        plt.savefig(f"/1_scPRINT/fetalSC_figures/enrichment_plots/enrichment_plot_{selected_community}.png",
                    dpi=300, bbox_inches='tight')
        plt.show()
        
    except ValueError as e:
        if "No enrich terms when cutoff" in str(e):
            print(f"Skipping community {selected_community}: No enriched terms found at cutoff=0.25")
        else:
            print(f"Skipping community {selected_community}: {str(e)}")
        continue
    except Exception as e:
        print(f"Skipping community {selected_community}: Unexpected error - {str(e)}")
        continue

## Comparing with add-on programs from NichCompass analysis

In [ ]:
# Find interesting genes to focus on (e.g., stem cell markers, developmental genes)
stem_cell_markers = [
    'INSM1', 'HES6', 'PROX1', 
    'SOX4','SPIB', 'SOX9', 
    "FOXA3","IRF8", "HMGP2", "L1TD1"] # genes from add-on programs

print("Processing stem cell markers:")
print(f"Total markers to check: {len(stem_cell_markers)}")

# Create directory for saving plots if it doesn't exist
marker_plots_dir = "/1_scPRINT/fetalSC_figures/marker_networks"
os.makedirs(marker_plots_dir, exist_ok=True)

# Process each marker gene individually
for focus_gene in stem_cell_markers:
    if focus_gene not in grn_full.var.index:
        print(f"Skipping {focus_gene}: Gene not found in network")
        continue
    
    try:
        print(f"Analyzing network around {focus_gene}...")
        
        # Create a subgraph for this specific gene
        G_focus = grn_full.plot_subgraph(focus_gene, only=50, max_genes=30, interactive=False)
        
        if len(G_focus.nodes) == 0:
            print(f"Skipping {focus_gene}: No network connections found")
            continue
        
        plt.figure(figsize=(12, 10))

        # Use a layout algorithm for better visualization
        pos = nx.spring_layout(G_focus, k=1, iterations=50)

        # Draw the network
        nx.draw(G_focus, pos,
                with_labels=True,
                node_color='lightblue',
                node_size=500,
                font_size=8,
                font_weight='bold',
                arrows=True,
                edge_color='gray',
                alpha=0.7)

        # Highlight the focus gene if it exists in the network
        if focus_gene in G_focus.nodes:
            pos_focus = pos[focus_gene]
            nx.draw_networkx_nodes(G_focus, pos, nodelist=[focus_gene], 
                                 node_color='red', node_size=800, alpha=0.9)

        plt.title(f'Gene Regulatory Network - {focus_gene}',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Save as PNG
        filename = f"grn_marker_{focus_gene}_network.png"
        filepath = os.path.join(marker_plots_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved: {filepath}")
        plt.close()
        
    except Exception as e:
        print(f"Error processing {focus_gene}: {str(e)}")
        continue

print(f"\nCompleted processing stem cell markers")